# Cell 1: Project Overview

# Temporal Reference-Guided Diffusion Network (TRDN) for Video Dehazing

Colab/VS Code notebook for TRDN V1 on REVIDE. The pipeline uses 10 hazy frames, RAFT alignment, ConvLSTM temporal memory, reference selection, and Stable Diffusion inpainting conditioning. No training results are precomputed or faked.


# Cell 2: Install Dependencies


In [ ]:
import sys, subprocess
packages = ["-r", "requirements_colab.txt"]
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
except Exception:
    fallback = ["diffusers>=0.30.0", "transformers>=4.41.0", "accelerate>=0.31.0", "tensorboard>=2.16.0", "lpips>=0.1.4", "torchmetrics>=1.4.0", "opencv-python>=4.8.0", "scikit-image>=0.22.0", "matplotlib>=3.7.0", "tqdm>=4.66.0"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *fallback])
print("Dependencies installed or already available.")


# Cell 3: Mount Google Drive


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB = True
except Exception as exc:
    COLAB = False
    print("Drive mount skipped:", repr(exc))
PROJECT_ROOT = Path('/content/drive/MyDrive/TRDN_REVIDE') if COLAB else Path.cwd() / 'TRDN_REVIDE'
print("PROJECT_ROOT:", PROJECT_ROOT)


# Cell 4: Configuration


In [ ]:
import sys, os, json, torch
from pathlib import Path
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").exists(): sys.path.insert(0, str(REPO_ROOT))
if (REPO_ROOT.parent / "src").exists(): sys.path.insert(0, str(REPO_ROOT.parent))
from src.config import TRDNConfig
DATASET_ROOT = "/content/drive/MyDrive/REVIDE"  # EDIT THIS
cfg = TRDNConfig(dataset_root=DATASET_ROOT, project_root=str(PROJECT_ROOT), image_size=256, crop_size=256, seq_len=10, batch_size=1, num_workers=2, mixed_precision="fp16")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(json.dumps(cfg.to_dict(), indent=2, default=str))
print("DEVICE:", DEVICE, "DATASET_ROOT exists:", Path(cfg.dataset_root).exists())


# Cell 5: Directory Creation


In [ ]:
paths = cfg.ensure_dirs()
for name, path in paths.items(): print(name, "->", path)


# Cell 6: Imports


In [ ]:
import random, math, numpy as np, matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from tqdm.auto import tqdm
from accelerate.utils import set_seed
from src.dataset import REVIDESequenceDataset
from src.masks import generate_haze_mask
from src.haze import simulate_realistic_haze
from src.flow import load_raft, compute_warped_references_batch, flow_to_rgb
from src.warp import warp_with_flow
from src.convlstm import TemporalMemoryModule
from src.reference_selector import ReferenceSelectionModule
from src.diffusion_adapter import TemporalConditioningAdapter, load_diffusion_backbone, enable_lora_for_unet
from src.losses import LossBundle
from src.metrics import psnr_metric, ssim_metric
from src.train import dry_run_shape_test, train_trdn, make_dataloaders, build_temporal_modules
from src.validate import infer_dehazed_batch, validate_trdn
set_seed(cfg.seed)
def show_tensor_images(images, titles, figsize=(16,4)):
    plt.figure(figsize=figsize)
    for i,(img,title) in enumerate(zip(images,titles),1):
        x = img.detach().float().cpu()
        if x.ndim == 4: x = x[0]
        plt.subplot(1,len(images),i)
        if x.shape[0] == 1: plt.imshow(x[0].clamp(0,1), cmap='magma')
        else: plt.imshow(x.clamp(0,1).permute(1,2,0))
        plt.title(title); plt.axis('off')
    plt.tight_layout(); plt.show()
print("Imports ready.")


# Cell 7: Dataset Loader


In [ ]:
train_dataset = REVIDESequenceDataset(cfg.dataset_root, cfg.train_split, cfg.seq_len, cfg.crop_size, True, cfg.image_extensions, True)
val_dataset = REVIDESequenceDataset(cfg.dataset_root, cfg.val_split, cfg.seq_len, cfg.crop_size, False, cfg.image_extensions, True)
print("train samples:", len(train_dataset), "val samples:", len(val_dataset))
sample = train_dataset[0]
for key in ["frames", "current_frame", "target_frame", "mask", "corrupted_frame", "warped_references"]:
    print(key, tuple(sample[key].shape), sample[key].dtype, float(sample[key].min()), float(sample[key].max()))


# Cell 8: Mask Generation


In [ ]:
masks = [generate_haze_mask(cfg.image_size, cfg.image_size, mode) for mode in ["rectangle", "ellipse", "blob", "perlin"]]
show_tensor_images(masks, ["rectangle", "ellipse", "blob", "perlin"])


# Cell 9: Haze Simulation


In [ ]:
H = W = cfg.image_size
yy, xx = torch.meshgrid(torch.linspace(0,1,H), torch.linspace(0,1,W), indexing='ij')
clean = torch.stack([xx, yy, 0.5*xx + 0.5*yy], dim=0)
mask = generate_haze_mask(H, W, "perlin")
corrupted = simulate_realistic_haze(clean.unsqueeze(0), mask.unsqueeze(0))[0]
show_tensor_images([clean, mask, corrupted], ["clean", "mask", "synthetic haze"])


# Cell 10: RAFT Flow


In [ ]:
RAFT_MODEL = None
if cfg.use_raft_alignment and torch.cuda.is_available():
    RAFT_MODEL = load_raft(DEVICE, cfg.freeze_raft)
    print("Loaded frozen pretrained RAFT.")
else:
    print("RAFT skipped. CUDA unavailable or cfg.use_raft_alignment=False.")


# Cell 11: Warping


In [ ]:
frames = sample["frames"].unsqueeze(0).to(DEVICE)
warped_refs, flows = compute_warped_references_batch(frames, RAFT_MODEL)
print("frames", tuple(frames.shape), "warped_refs", tuple(warped_refs.shape), "flows", tuple(flows.shape))
show_tensor_images([frames[0,0].cpu(), flow_to_rgb(flows[0,0].cpu()), warped_refs[0,0].cpu(), frames[0,-1].cpu()], ["prev", "flow", "warped", "current"])


# Cell 12: Dataset Visualization


In [ ]:
show_tensor_images([sample["frames"][0], sample["frames"][-1], sample["target_frame"], sample["mask"], sample["corrupted_frame"]], ["t-9 hazy", "current hazy", "clean target", "mask", "corrupted"])
print("sequence:", sample["sequence_name"])
print("first frame paths:", sample["frame_paths"][:3])


# Cell 13: ConvLSTM Module


In [ ]:
temporal_memory = TemporalMemoryModule(hidden_dim=64).to(DEVICE)
aligned_frames = torch.cat([warped_refs, frames[:, -1:].contiguous()], dim=1)
with torch.no_grad(): memory = temporal_memory(aligned_frames)
print("aligned_frames", tuple(aligned_frames.shape), "memory", tuple(memory.shape))
mem_vis = memory[0,:1].cpu(); mem_vis = (mem_vis - mem_vis.min()) / (mem_vis.max() - mem_vis.min() + 1e-8)
show_tensor_images([mem_vis], ["ConvLSTM memory ch0"])


# Cell 14: Reference Selection Module


In [ ]:
reference_selector = ReferenceSelectionModule(num_references=cfg.seq_len-1).to(DEVICE)
with torch.no_grad(): ref = reference_selector(warped_refs, memory)
print("weights", tuple(ref["weights"].shape), "weighted_reference", tuple(ref["weighted_reference"].shape), "reference_feature", tuple(ref["reference_feature"].shape))
show_tensor_images([ref["weights"][0,i:i+1].cpu() for i in range(3)], ["ref0", "ref1", "ref2"])


# Cell 15: Diffusion Backbone


In [ ]:
LOAD_DIFFUSION_NOW = False  # Set True when ready to download/load Stable Diffusion.
DIFFUSION = None
if LOAD_DIFFUSION_NOW:
    DIFFUSION = load_diffusion_backbone(cfg, DEVICE)
    print("UNet in_channels:", DIFFUSION["unet"].config.in_channels, "cross_attention_dim:", DIFFUSION["unet"].config.cross_attention_dim)
else:
    print("Diffusion loading deferred. Set LOAD_DIFFUSION_NOW=True to load runwayml/stable-diffusion-inpainting.")


# Cell 16: Conditioning Adapter


In [ ]:
cross_attention_dim = 768 if DIFFUSION is None else DIFFUSION["unet"].config.cross_attention_dim
conditioning_adapter = TemporalConditioningAdapter(cross_attention_dim=cross_attention_dim, num_tokens=16).to(DEVICE)
with torch.no_grad(): tokens = conditioning_adapter(memory, ref["reference_feature"])
print("conditioning tokens", tuple(tokens.shape))


# Cell 17: Loss Functions


In [ ]:
loss_bundle = LossBundle(device=DEVICE)
print("LPIPS enabled:", loss_bundle.lpips_model is not None)
print("PSNR self-test:", psnr_metric(sample["target_frame"], sample["target_frame"]))
print("SSIM self-test:", ssim_metric(sample["target_frame"], sample["target_frame"]))


# Cell 18: Training Loop


In [ ]:
# Training is real but disabled by default to avoid accidental long Colab runs.
cfg.run_training_now = False
if cfg.run_training_now:
    result = train_trdn(cfg)
    print(result)
else:
    print("Set cfg.run_training_now=True to train. Checkpoints/logs go to", cfg.project_root)


# Cell 19: Validation


In [ ]:
cfg.run_validation_now = False
if cfg.run_validation_now:
    if DIFFUSION is None: DIFFUSION = load_diffusion_backbone(cfg, DEVICE)
    tm, rs, ca = build_temporal_modules(cfg, DIFFUSION["unet"].config.cross_attention_dim, DEVICE)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)
    metrics = validate_trdn(val_loader, DIFFUSION, tm, rs, ca, loss_bundle, DEVICE, raft_model=RAFT_MODEL, max_batches=4, num_steps=10)
    print({k:v for k,v in metrics.items() if isinstance(v, float)})
else:
    print("Set cfg.run_validation_now=True after loading/training a checkpoint.")


# Cell 20: Inference


In [ ]:
cfg.run_inference_now = False
if cfg.run_inference_now:
    if DIFFUSION is None: DIFFUSION = load_diffusion_backbone(cfg, DEVICE)
    output = infer_dehazed_batch(frames, sample["mask"].unsqueeze(0).to(DEVICE), sample["corrupted_frame"].unsqueeze(0).to(DEVICE), DIFFUSION, temporal_memory, reference_selector, conditioning_adapter, DEVICE, raft_model=RAFT_MODEL, num_steps=cfg.num_inference_steps)
    show_tensor_images([sample["corrupted_frame"], sample["target_frame"], output["prediction"][0].cpu(), output["weighted_reference"][0].cpu()], ["input", "target", "prediction", "weighted ref"])
else:
    print("Set cfg.run_inference_now=True after loading diffusion/checkpoint.")


# Cell 21: Debugging Tools


In [ ]:
shape_report = dry_run_shape_test(seq_len=cfg.seq_len, image_size=64, batch_size=1)
print("Dry-run shape test:")
for key, value in shape_report.items(): print(f"{key}: {value}")
expected = {"frames": (1,10,3,64,64), "current_hazy": (1,3,64,64), "target_clean": (1,3,64,64), "mask": (1,1,64,64), "warped_references": (1,9,3,64,64)}
for key, value in expected.items(): assert shape_report[key] == value, (key, shape_report[key], value)
print("All expected tensor shapes verified.")


# Cell 22: Future Transformer Integration Placeholder


In [ ]:
import torch.nn as nn
class TemporalRetrievalTransformer(nn.Module):
    def __init__(self, feature_dim=64, token_dim=256, num_layers=4, num_heads=8):
        super().__init__()
        self.patch_embed = nn.Conv2d(feature_dim, token_dim, kernel_size=4, stride=4)
        layer = nn.TransformerEncoderLayer(d_model=token_dim, nhead=num_heads, dim_feedforward=token_dim*4, dropout=0.1, activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(token_dim)
    def forward(self, temporal_feature_map):
        tokens = self.patch_embed(temporal_feature_map).flatten(2).transpose(1,2)
        return self.norm(self.encoder(tokens))
future_transformer = TemporalRetrievalTransformer().to(DEVICE)
with torch.no_grad(): future_tokens = future_transformer(torch.rand(1,64,64,64,device=DEVICE))
print("future transformer tokens", tuple(future_tokens.shape))
